# 전처리 전후 데이터 통계량 표 생성

이 노트북은 다음 두 파일을 기준으로 3개 표를 생성합니다.

- 전: `data/processed/토크나이징_전_전처리.csv`의 `cleaned_text`
- 후: `data/processed/최종정제_v2.csv`의 `final_text`

생성 표:

1. 전처리 전 후 비교
2. 글자 수 통계량
3. 형태소 수 통계량


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
BEFORE_PATH = ROOT / "data" / "processed" / "토크나이징_전_전처리.csv"
AFTER_PATH = ROOT / "data" / "processed" / "최종정제_v2.csv"
OUTPUT_DIR = ROOT / "results" / "eda_stats"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("전 파일:", BEFORE_PATH)
print("후 파일:", AFTER_PATH)
print("저장 폴더:", OUTPUT_DIR)

전 파일: c:\Users\user\Desktop\4-2\사사_텍분\data\processed\토크나이징_전_전처리.csv
후 파일: c:\Users\user\Desktop\4-2\사사_텍분\data\processed\최종정제_v2.csv
저장 폴더: c:\Users\user\Desktop\4-2\사사_텍분\results\eda_stats


In [2]:
before = pd.read_csv(BEFORE_PATH, encoding="utf-8-sig", low_memory=False)
after = pd.read_csv(AFTER_PATH, encoding="utf-8-sig")

before_text = before["cleaned_text"].fillna("").astype(str)
after_text = after["final_text"].fillna("").astype(str)

print("전 rows:", len(before), "columns:", list(before.columns))
print("후 rows:", len(after), "columns:", list(after.columns))

전 rows: 74356 columns: ['cid', 'text', 'time', 'author', 'channel', 'votes', 'replies', 'photo', 'heart', 'reply', 'time_parsed', 'paid', 'cleaned_text']
후 rows: 55376 columns: ['cid', 'final_text']


In [3]:
def token_counts(series: pd.Series) -> pd.Series:
    return series.apply(lambda s: len(s.split()) if s.strip() else 0)


def char_counts(series: pd.Series) -> pd.Series:
    return series.apply(len)


def unique_word_count(series: pd.Series) -> int:
    vocab = set()
    for text in series:
        if text.strip():
            vocab.update(text.split())
    return len(vocab)


def change_rate(before_value: float, after_value: float) -> float:
    if before_value == 0:
        return float("nan")
    return (after_value - before_value) / before_value * 100


before_tokens = token_counts(before_text)
after_tokens = token_counts(after_text)
before_chars = char_counts(before_text)
after_chars = char_counts(after_text)

In [4]:
comparison_rows = [
    ("총 댓글 수", len(before), len(after), "count"),
    ("전체 단어 수", int(before_tokens.sum()), int(after_tokens.sum()), "count"),
    ("고유 단어 수", unique_word_count(before_text), unique_word_count(after_text), "count"),
    ("전체 글자 수", int(before_chars.sum()), int(after_chars.sum()), "count"),
    ("댓글 당 평균 단어 수", before_tokens.mean(), after_tokens.mean(), "float"),
    ("댓글 단어 수 중앙값", before_tokens.median(), after_tokens.median(), "float"),
    ("댓글 당 평균 글자 수", before_chars.mean(), after_chars.mean(), "float"),
    ("댓글 글자 수 중앙값", before_chars.median(), after_chars.median(), "float"),
]

comparison_table = pd.DataFrame(
    [
        {
            "지표": name,
            "전": before_value,
            "후": after_value,
            "변화율": change_rate(before_value, after_value),
        }
        for name, before_value, after_value, _ in comparison_rows
    ]
)

comparison_display = comparison_table.copy()
for idx, (_, _, _, kind) in enumerate(comparison_rows):
    if kind == "count":
        comparison_display.loc[idx, "전"] = f"{int(comparison_table.loc[idx, '전']):,}"
        comparison_display.loc[idx, "후"] = f"{int(comparison_table.loc[idx, '후']):,}"
    else:
        comparison_display.loc[idx, "전"] = f"{comparison_table.loc[idx, '전']:.2f}"
        comparison_display.loc[idx, "후"] = f"{comparison_table.loc[idx, '후']:.2f}"
comparison_display["변화율"] = comparison_table["변화율"].map(lambda x: f"{x:.1f}%")

comparison_table.to_csv(OUTPUT_DIR / "preprocess_comparison_raw.csv", index=False, encoding="utf-8-sig")
comparison_display.to_csv(OUTPUT_DIR / "preprocess_comparison_display.csv", index=False, encoding="utf-8-sig")

comparison_display

C:\Users\user\AppData\Local\Temp\ipykernel_2240\3633289561.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '74,356' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  comparison_display.loc[idx, "전"] = f"{int(comparison_table.loc[idx, '전']):,}"
C:\Users\user\AppData\Local\Temp\ipykernel_2240\3633289561.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '55,376' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  comparison_display.loc[idx, "후"] = f"{int(comparison_table.loc[idx, '후']):,}"


,지표,전,후,변화율
0,총 댓글 수,"74,356","55,376",-25.5%
1,전체 단어 수,"1,452,073","697,053",-52.0%
2,고유 단어 수,"258,969","4,096",-98.4%
3,전체 글자 수,"5,924,139","2,288,892",-61.4%
4,댓글 당 평균 단어 수,19.53,12.59,-35.5%
5,댓글 단어 수 중앙값,8.00,6.00,-25.0%
6,댓글 당 평균 글자 수,79.67,41.33,-48.1%
7,댓글 글자 수 중앙값,34.00,20.00,-41.2%


In [5]:
def describe_table(counts: pd.Series) -> pd.DataFrame:
    desc = counts.describe(percentiles=[0.25, 0.5, 0.75])
    return pd.DataFrame(
        {
            "통계량": ["개수", "평균", "표준편차", "최솟값", "25%", "50%", "75%", "최댓값"],
            "값": [
                desc["count"],
                desc["mean"],
                desc["std"],
                desc["min"],
                desc["25%"],
                desc["50%"],
                desc["75%"],
                desc["max"],
            ],
        }
    )


def format_describe_table(table: pd.DataFrame) -> pd.DataFrame:
    display_table = table.copy()
    integer_stats = {"개수", "최솟값", "25%", "50%", "75%", "최댓값"}
    display_table["값"] = display_table.apply(
        lambda row: f"{int(row['값']):,}" if row["통계량"] in integer_stats else f"{row['값']:.2f}",
        axis=1,
    )
    return display_table

In [6]:
char_stats_table = describe_table(after_chars)
char_stats_display = format_describe_table(char_stats_table)

char_stats_table.to_csv(OUTPUT_DIR / "final_text_char_stats_raw.csv", index=False, encoding="utf-8-sig")
char_stats_display.to_csv(OUTPUT_DIR / "final_text_char_stats_display.csv", index=False, encoding="utf-8-sig")

char_stats_display

,통계량,값
0,개수,"55,376"
1,평균,41.33
2,표준편차,66.21
3,최솟값,5
4,25%,10
5,50%,20
6,75%,46
7,최댓값,"2,055"


In [7]:
morph_stats_table = describe_table(after_tokens)
morph_stats_display = format_describe_table(morph_stats_table)

morph_stats_table.to_csv(OUTPUT_DIR / "final_text_morph_stats_raw.csv", index=False, encoding="utf-8-sig")
morph_stats_display.to_csv(OUTPUT_DIR / "final_text_morph_stats_display.csv", index=False, encoding="utf-8-sig")

morph_stats_display

,통계량,값
0,개수,"55,376"
1,평균,12.59
2,표준편차,19.76
3,최솟값,2
4,25%,3
5,50%,6
6,75%,14
7,최댓값,618


In [8]:
print("저장 완료:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path)

저장 완료:
- c:\Users\user\Desktop\4-2\사사_텍분\results\eda_stats\final_text_char_stats_display.csv
- c:\Users\user\Desktop\4-2\사사_텍분\results\eda_stats\final_text_char_stats_raw.csv
- c:\Users\user\Desktop\4-2\사사_텍분\results\eda_stats\final_text_morph_stats_display.csv
- c:\Users\user\Desktop\4-2\사사_텍분\results\eda_stats\final_text_morph_stats_raw.csv
- c:\Users\user\Desktop\4-2\사사_텍분\results\eda_stats\preprocess_comparison_display.csv
- c:\Users\user\Desktop\4-2\사사_텍분\results\eda_stats\preprocess_comparison_raw.csv
